In [1]:
import time
import random
import string
import tiktoken
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# [설정 영역]
# ==========================================
BASE_URL   = "http://localhost:8000/v1"  # vLLM 기본 포트
API_KEY    = "EMPTY"                      # vLLM은 API key 미검증
MODEL_NAME = "/home/dev/models/Qwen3-8B-AWQ"          # vllm serve 시 사용한 모델 ID와 동일하게 설정

# 동시 요청 수 목록 — 각 값에 대해 독립적으로 벤치마크 수행
CONCURRENT_REQUESTS = [1, 2, 4, 16, 64]

# 각 (프롬프트 크기, 동시 요청 수) 조합당 반복 횟수
ITERATIONS = 10

TEST_CASES = [
    {"label": "128",  "target_tokens": 128,   "output_len": 300},
    {"label": "1K",   "target_tokens": 1024,  "output_len": 300},
    {"label": "4K",   "target_tokens": 4096,  "output_len": 300},
]


client    = OpenAI(base_url=BASE_URL, api_key=API_KEY)
# 범용적으로 Qwen 등 최신 모델과 토큰 수가 거의 유사한 cl100k_base 사용
# 실제 토큰 수는 서버 응답(chunk.usage.prompt_tokens)으로 정확히 수집됨
tokenizer = tiktoken.get_encoding("cl100k_base")

In [2]:
# ==========================================
# [프롬프트 생성]
# ==========================================
def generate_exact_prompt(target_tokens):
    """
    tiktoken을 사용하여 정확히 target_tokens 길이에 맞게 프롬프트를 생성합니다.
    캐시 무력화를 위해 매 호출마다 다른 nonce가 삽입됩니다.
    """
    if target_tokens < 10:
        return "hi " * (target_tokens // 2 + 1)

    nonce  = "".join(random.choices(string.ascii_letters + string.digits, k=16))
    prefix = f"System Nonce: {nonce}\n\n"
    prefix_tokens = tokenizer.encode(prefix)

    if target_tokens <= len(prefix_tokens):
        return prefix

    remaining_target = target_tokens - len(prefix_tokens)
    base_text   = "The deepseek coder is a powerful model for coding tasks and general reasoning benchmarks. It efficiently handles context and generation tasks with mixture of experts architecture. "
    base_tokens = tokenizer.encode(base_text)

    repeat_count    = (remaining_target // len(base_tokens)) + 1
    extended_tokens = (base_tokens * repeat_count)[:remaining_target]

    return tokenizer.decode(prefix_tokens + extended_tokens)

In [3]:
# ==========================================
# [단일 요청]
# ==========================================
def run_single_benchmark(case):
    """
    단일 요청을 실행하고 (prefill_t/s, decode_t/s, actual_prompt_tokens, token_count) 를 반환합니다.
    실패 시 (None, None, None, error_message) 를 반환합니다.
    """
    prompt = generate_exact_prompt(case["target_tokens"])

    try:
        start_time = time.time()

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=case["output_len"],
            stream=True,
            temperature=0.0,
            # [핵심] 서버에서 실제 연산한 토큰 수를 응답 마지막에 포함시킴
            stream_options={"include_usage": True}
        )

        first_token_time     = None
        token_count          = 0
        actual_prompt_tokens = case["target_tokens"]  # 실패 시 기본값

        for chunk in response:
            # usage 정보가 있는 마지막 청크 처리
            if chunk.usage:
                actual_prompt_tokens = chunk.usage.prompt_tokens
                continue

            if not chunk.choices:
                continue

            if chunk.choices[0].delta.content is not None:
                if first_token_time is None:
                    first_token_time = time.time()
                token_count += 1

        end_time = time.time()

        prefill_duration = (first_token_time - start_time) if first_token_time else 0
        decode_duration  = (end_time - first_token_time)   if first_token_time else 0

        # 서버가 반환한 실제(actual) 토큰 수로 prefill 속도 계산
        prefill_speed = actual_prompt_tokens / prefill_duration if prefill_duration > 0 else 0
        decode_speed  = (token_count - 1) / decode_duration if (decode_duration > 0 and token_count > 1) else 0

        return prefill_speed, decode_speed, actual_prompt_tokens, token_count

    except Exception as e:
        return None, None, None, f"Error: {str(e)[:50]}"

In [4]:
# ==========================================
# [동시 요청 배치]
# ==========================================
def run_concurrent_batch(case, n_concurrent):
    """
    n_concurrent 개의 요청을 동시에 실행합니다.

    반환값:
        results      : 성공한 요청들의 (prefill, decode, p_count, e_count) 리스트
        wall_elapsed : 배치 전체 경과 시간(초) — wall-clock 기준
        n_failed     : 실패한 요청 수
    """
    results  = []
    n_failed = 0

    wall_start = time.time()
    with ThreadPoolExecutor(max_workers=n_concurrent) as executor:
        futures = [executor.submit(run_single_benchmark, case) for _ in range(n_concurrent)]
        for future in as_completed(futures):
            prefill, decode, p_count, e_count = future.result()
            if prefill is None:
                n_failed += 1
            else:
                results.append((prefill, decode, p_count, e_count))
    wall_elapsed = time.time() - wall_start

    return results, wall_elapsed, n_failed

In [5]:
# ==========================================
# [메인 벤치마크]
# ==========================================
print(f"\n🚀 Concurrent Benchmark: {MODEL_NAME}")
print(f"📡 Server: {BASE_URL} | 🔄 Iterations: {ITERATIONS} | ⚡ Concurrency levels: {CONCURRENT_REQUESTS}")
print("-" * 121)
print(f"{'Prompt Size':<15} | {'Concurrency':>11} | {'Avg Prefill (t/s)':>17} | {'Avg Decode (t/s)':>16} | {'Avg Total (s)':>13} | {'Throughput (t/s)':>16} | {'Status'}")
print("-" * 121)

# Warm-up
print(f"{'Warm-up...':<15} | {'':>11} | {'...':>17} | {'...':>16} | {'...':>13} | {'...':>16} | Loading model")
try:
    client.chat.completions.create(
        model=MODEL_NAME, messages=[{"role": "user", "content": "hi"}], max_tokens=1
    )
except:
    pass

for case in TEST_CASES:
    for n_concurrent in CONCURRENT_REQUESTS:

        iter_prefills    = []
        iter_decodes     = []
        iter_totals      = []
        iter_throughputs = []
        any_error        = None

        for _ in range(ITERATIONS):
            results, wall_elapsed, n_failed = run_concurrent_batch(case, n_concurrent)

            if not results:  # 모든 요청 실패
                any_error = "All requests failed"
                break

            avg_prefill = sum(r[0] for r in results) / len(results)
            avg_decode  = sum(r[1] for r in results) / len(results)
            throughput  = sum(r[3] for r in results) / wall_elapsed

            iter_prefills.append(avg_prefill)
            iter_decodes.append(avg_decode)
            iter_totals.append(wall_elapsed)
            iter_throughputs.append(throughput)

            time.sleep(0.5)

        if any_error:
            print(f"{case['label']:<15} | {n_concurrent:>11} | {'Fail':>17} | {'Fail':>16} | {'Fail':>13} | {'Fail':>16} | {any_error}")
        else:
            final_prefill    = sum(iter_prefills)    / len(iter_prefills)
            final_decode     = sum(iter_decodes)     / len(iter_decodes)
            final_total      = sum(iter_totals)      / len(iter_totals)
            final_throughput = sum(iter_throughputs) / len(iter_throughputs)
            n_ok = n_concurrent - n_failed  # 마지막 배치 기준 성공 수
            status = "ok" if n_failed == 0 else f"ok"

            print(f"{case['label']:<15} | {n_concurrent:>11} | {final_prefill:>17.2f} | {final_decode:>16.2f} | {final_total:>13.2f} | {final_throughput:>16.2f} | {status}")

print("-" * 121)
print("✅ Benchmark Completed.")


🚀 Concurrent Benchmark: /home/dev/models/Qwen3-8B-AWQ
📡 Server: http://localhost:8000/v1 | 🔄 Iterations: 10 | ⚡ Concurrency levels: [1, 2, 4, 16, 64]
-------------------------------------------------------------------------------------------------------------------------
Prompt Size     | Concurrency | Avg Prefill (t/s) | Avg Decode (t/s) | Avg Total (s) | Throughput (t/s) | Status
-------------------------------------------------------------------------------------------------------------------------
Warm-up...      |             |               ... |              ... |           ... |              ... | Loading model
128             |           1 |            152.18 |            52.73 |          7.78 |            39.07 | ok
128             |           2 |             55.60 |            51.77 |          8.20 |            73.38 | ok
128             |           4 |             52.76 |            49.40 |          8.62 |           138.68 | ok
128             |          16 |             4